In [6]:
from pipeline import Pipeline
from pipeline import PipelineConfig

# Surface model
cfg = PipelineConfig(
      tickers=["XLF", "GS", "JPM", "BAC", "BRK", "C"],
      start_date="2008-01-01",
      end_date="2025-04-30",
      model_kind="surface",
      signal_model_kind="risk_reversal",
      signal_factor_model="pca",
      signal_direction="auto",
      signal_entry_z=1.25,
      signal_exit_z=0.30,
      signal_min_liquidity_points=150.0,
      signal_max_fit_rmse_iv=0.60,
      signal_cross_section_top_k=2,
      signal_cross_section_bottom_k=2,
      signal_max_name_weight=0.35,
      signal_max_gross_leverage=1.5,
      signal_vol_target_daily=0.02,
      backtest_rr_put_log_moneyness=-0.15,
      backtest_rr_call_log_moneyness=0.15,
      backtest_roll_threshold_days=5,
  )


loaded = Pipeline(cfg).run_load()
processed = Pipeline(cfg).run_process(loaded)
filename = "test_model.parquet"
modeled = Pipeline(cfg).run_model(processed, save=True, saved=filename)#, saved=path)

# Smile model
cfg_smile = PipelineConfig(**{**cfg.__dict__, "model_kind": "smile"})
p_smile = Pipeline(cfg_smile)
filename_smile = "test_smile.parquet"
modeled_smile = p_smile.run_model(processed, save=True, saved=filename_smile)#, saved="test_smile.parquet")


SSVI tickers:   0%|          | 0/6 [00:00<?, ?it/s]

SSVI calibration error | ticker=XLF date=2016-08-30 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.


SSVI calibration error | ticker=XLF date=2016-09-15 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.
SSVI calibration error | ticker=XLF date=2016-09-21 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.
SSVI calibration error | ticker=XLF date=2016-09-23 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.
SSVI calibration error | ticker=XLF date=2016-09-26 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.


SSVI tickers:  67%|██████▋   | 4/6 [48:43<22:43, 681.72s/it]  

SSVI calibration error | ticker=BRK date=2009-06-24 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.
SSVI calibration error | ticker=BRK date=2009-06-25 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.


SSVI tickers:  83%|████████▎ | 5/6 [56:47<10:10, 610.71s/it]

SSVI calibration error | ticker=C date=2009-01-16 backend=auto error=ValueError: Need at least 2 maturities for SSVI calibration.


Smile tickers: 100%|██████████| 6/6 [00:22<00:00,  3.80s/it]


In [ ]:
# Volatility surface
from analytics.volatility_models import plot_surface_fit_analytics_from_output
metrics = plot_surface_fit_analytics_from_output(modeled, ticker="GS")

# Volatility smile
from analytics.volatility_models import plot_smile_fit_analytics_from_output
metrics = plot_smile_fit_analytics_from_output(modeled_smile, ticker="GS")


Surface fit summary:
              rmse          mae         bias  max_abs_error           r2
count  4360.000000  4360.000000  4360.000000    4360.000000  4360.000000
mean      0.117242     0.063400     0.030395       0.734866     0.779996
std       0.072490     0.036046     0.030690       0.470901     0.093132
min       0.016049     0.007391    -0.059103       0.096336     0.318169
25%       0.065305     0.037410     0.008580       0.385186     0.728135
50%       0.097118     0.053419     0.022489       0.603391     0.795095
75%       0.150542     0.080205     0.044912       0.961286     0.846813
max       0.645443     0.290121     0.229304       3.418014     0.981920


In [ ]:
# Skew
from analytics.skew_models import plot_skew_analytics
print(modeled)
skew = Pipeline(cfg).run_skew(modeled, save=True, saved="test_skew.parquet")
plot_skew_analytics(skew)




ModelOutput(model_by_ticker={'XLF':           date ticker      skew  target_t   k0       rho       eta     gamma  \
0   2008-01-02    XLF -0.752572  0.082192  0.0 -0.531933  0.936773  0.468109   
1   2008-01-03    XLF -0.627569  0.082192  0.0 -0.534075  1.254990  0.361182   
2   2008-01-04    XLF -0.791493  0.082192  0.0 -0.522028  1.150844  0.436909   
3   2008-01-07    XLF -1.021311  0.082192  0.0 -0.539343  0.727103  0.587553   
4   2008-01-08    XLF -0.890205  0.082192  0.0 -0.476866  0.827585  0.557844   
..         ...    ...       ...       ...  ...       ...       ...       ...   
582 2010-04-26    XLF -0.891525  0.082192  0.0 -0.579863  0.726139  0.539835   
583 2010-04-27    XLF -1.196291  0.082192  0.0 -0.562570  0.652414  0.631694   
584 2010-04-28    XLF -1.201321  0.082192  0.0 -0.606338  0.476579  0.681767   
585 2010-04-29    XLF -1.115883  0.082192  0.0 -0.632709  0.563371  0.623860   
586 2010-04-30    XLF -1.516913  0.082192  0.0 -0.622409  0.706180  0.647714   

   

,ticker,n_dates,mean_skew,std_skew,mean_abs_skew,acf_lag1,acf_lag5,half_life_days,mean_abs_delta,sign_flip_rate,mean_n_points,mean_rmse_implied_volatility
0,BAC,587,-0.778719,0.270165,0.778719,0.617402,0.534977,1.437366,0.148493,0.000000,182.652470,0.159625
1,BRK,213,-0.655302,0.345181,0.655302,0.626303,0.459282,1.481334,0.197913,0.000000,106.788732,0.046201
2,C,586,-0.644998,0.297970,0.645014,0.711505,0.575253,2.036439,0.160055,0.003419,111.131399,0.171826
3,GS,587,-0.888006,0.533525,0.888006,0.710235,0.652431,2.025801,0.199577,0.000000,590.833049,0.138058
4,JPM,587,-0.885129,0.253094,0.885129,0.437728,0.346390,0.839001,0.169901,0.000000,230.478705,0.139932
5,XLF,587,-0.771457,0.225552,0.771457,0.508795,0.420218,1.025805,0.154298,0.000000,223.056218,0.129178


In [ ]:

from analytics.backtest_models import plot_backtest_analytics, pnl_attribution
signals = Pipeline(cfg).run_signals(skew)
backtest = Pipeline(cfg).run_backtest(signals, skew)
plot_backtest_analytics(backtest)
pnl_attribution(backtest)



Backtest summary:
   n_days  avg_daily_net_pnl  std_daily_net_pnl  daily_sharpe_like  \
0   587.0           0.077413          10.540744           0.116585   

   daily_sortino_like  final_cum_net_pnl  max_drawdown  hit_rate  \
0            0.219905          45.441228   -182.700945  0.155026   

   avg_daily_turnover  total_option_pnl  total_hedge_pnl  total_cost  
0            0.005893       -128.351349       270.804082   237.77804  
PnL component totals:
     component   total_pnl  vs_net_multiple
0   option_leg -128.351349        -2.824557
1    hedge_leg  270.804082         5.959436
2    skew_vega   90.594785         1.993669
3   level_vega   10.686103         0.235163
4        gamma  -22.836597        -0.502552
5        theta   -1.432151        -0.031517
6  unexplained  176.056887         3.874387
7         cost -237.778040        -5.232650
8          net   45.441228         1.000000
Worst 10 days (PnL drivers):
          date  portfolio_net_pnl  portfolio_option_pnl  portfolio_hedg

,ticker,option_pnl,hedge_pnl,cost,net_pnl,trades,avg_abs_position,recon_error
0,JPM,-962.998918,1187.943002,300.325005,-170.504586,42,0.016912,9.512366e+01
1,BRK,-7.365663,6.313226,42.574935,-43.627372,12,0.004724,0.000000e+00
2,C,0.000000,0.000000,0.000000,0.000000,0,0.000000,0.000000e+00
3,BAC,-1049.764164,601.590005,97.248821,16.906711,17,0.006588,-5.623297e+02
4,GS,1408.089010,-522.745173,563.397504,321.946334,69,0.015494,1.705303e-13


In [ ]:
print(backtest.summary)

{'n_days': 587.0, 'avg_daily_net_pnl': 0.0774126550199246, 'std_daily_net_pnl': 10.540743846230097, 'daily_sharpe_like': 0.11658454270390517, 'final_cum_net_pnl': 45.44122849669568}
